# MCMC Probabilistic State Machines — Stochastic Agentic Workflows

**Multigen SDK — Notebook 16**

---

In this notebook we explore one of the most powerful and underused patterns in agentic AI system design:
**probabilistic state machines driven by Markov Chain Monte Carlo (MCMC) sampling**.

Unlike deterministic pipelines (A → B → C, always), MCMC state machines allow agents to:
- **Revisit earlier stages** when quality is insufficient
- **Explore alternative paths** through sampling
- **Converge toward high-quality outputs** via adaptive transitions
- **Run in parallel ensembles** and pick the best trajectory

This is how real iterative creative and analytical workflows behave — not as rigid pipelines, but as guided explorations.

---

### What You Will Learn

| Section | Concept |
|---|---|
| 1 | Why deterministic workflows fail for iterative AI tasks |
| 2 | Markov Chains 101 — states, transitions, probabilities |
| 3 | MCMC in agentic systems — temperature, samplers, convergence |
| 4 | Building a 3-state writing machine |
| 5 | Transition probabilities |
| 6 | Running the machine — result.path, result.status |
| 7 | Self-loops — revision behavior |
| 8 | MCMC samplers — weighted, greedy, metropolis |
| 9 | Temperature effect |
| 10 | Adaptive transitions |
| 11 | transition_matrix() |
| 12 | run_ensemble() — parallel chains |
| 13 | Convergence detection |
| 14 | On-enter/on-exit hooks |
| 15 | Real-world: Research refinement machine |

## Section 1: Why Deterministic Workflows Fail for Iterative AI Tasks

Most AI orchestration frameworks assume a **linear, deterministic** execution model:

```
Input → Agent A → Agent B → Agent C → Output
```

This works well for **transformation pipelines** (extract → summarize → format), but breaks down for **quality-sensitive, iterative** tasks like:

- **Writing and editing**: A first draft rarely meets quality thresholds. You need revision loops.
- **Research synthesis**: Searching for evidence may require re-analyzing if coverage is insufficient.
- **Code generation**: Generated code may fail tests, requiring regeneration with different strategies.
- **Multi-criteria optimization**: You want to explore diverse approaches before committing to one.

### The Problem with Hard Determinism

Consider a writing pipeline:

```
draft_agent → review_agent → publish_agent
```

**What happens when the draft is bad?** In a deterministic chain:
1. `draft_agent` produces a poor draft.
2. `review_agent` flags it as insufficient.
3. `publish_agent` still runs and publishes garbage.

You could add conditional logic (`if score < 0.8: retry`), but this leads to:
- Spaghetti control flow with nested conditionals
- Infinite loops with no principled termination criterion
- No way to model **probabilistic transitions** (sometimes the draft is good enough)

### The Solution: Probabilistic State Machines

Instead of hard conditionals, we model the workflow as a **Markov Chain** where:
- Each **state** is an agent (draft, review, final)
- Each **transition** has a **probability** based on output quality
- The system **samples** from possible next states
- Termination is governed by **convergence criteria**

This is how Markov Chain Monte Carlo (MCMC) reasoning works — and it maps naturally to iterative AI workflows.

## Section 2: Markov Chains 101 — States, Transitions, Probabilities

A **Markov Chain** is a mathematical system that transitions between states where:
- The probability of going to state `j` depends **only on the current state** `i` (the Markov property)
- Each state has outgoing transition probabilities that sum to 1.0

### Formal Definition

Given states `S = {s1, s2, ..., sn}`, the transition matrix `P` is defined as:

```
P[i][j] = P(next state = sj | current state = si)

For all i: sum_j P[i][j] = 1.0  (rows sum to 1)
```

### ASCII Diagram: A Simple 3-State Writing Chain

```
                    0.2 (self-loop: needs more work)
                   ┌─────┐
                   │     ▼
  START ──────► [DRAFT] ──── 0.8 ────► [REVIEW] ──── 0.9 ────► [FINAL]
                                           │                      ▲
                                           │ 0.1 (back to draft) ─┘
                                           └─────────────────────────
                                                (actually goes back)
```

Reading the diagram:
- From **DRAFT**: 80% chance → REVIEW, 20% chance → back to DRAFT (revision needed)
- From **REVIEW**: 90% chance → FINAL, 10% chance → back to DRAFT (major issues)
- **FINAL** is an absorbing state (probability 1.0 of staying terminal)

### Transition Matrix for This Chain

```
         DRAFT  REVIEW  FINAL
DRAFT  [  0.2    0.8     0.0  ]
REVIEW [  0.1    0.0     0.9  ]
FINAL  [  0.0    0.0     1.0  ]  ← absorbing state
```

### Monte Carlo Sampling

At each step, we **sample** from the transition distribution:
- At state DRAFT: draw random number `r ~ Uniform(0,1)`
- If `r < 0.2`: stay in DRAFT (another revision)
- If `r >= 0.2`: go to REVIEW

This sampling gives us **stochastic trajectories** — different runs may follow different paths, which is exactly what we want for exploring the quality landscape.

## Section 3: MCMC in Agentic Systems

In the Multigen SDK, **StateMachine** implements MCMC-driven agentic workflows with three key parameters:

### Temperature

Temperature `T` modulates how much the sampler explores vs. exploits:

- **T = 0.0 (greedy)**: Always take the highest-probability transition. Deterministic, no exploration.
- **T = 1.0 (balanced)**: Sample according to the raw probabilities. Standard behavior.
- **T > 1.0 (high entropy)**: Flatten the probability distribution, increase exploration of unlikely paths.

The temperature-scaled probability for transition `i → j` is:

```
P_T(i → j) = exp(log(P(i→j)) / T) / sum_k exp(log(P(i→k)) / T)
```

This is the **softmax with temperature** — the same concept used in LLM sampling!

### Samplers

The SDK provides three sampler strategies:

| Sampler | Description | Best For |
|---|---|---|
| `weighted` | Sample proportional to transition probabilities | General use, balanced exploration |
| `greedy` | Always pick the highest-probability transition | Stable, predictable workflows |
| `metropolis` | Metropolis-Hastings accept/reject step | Theoretical correctness, rare transitions |

### Convergence

MCMC runs until one of:
1. An **absorbing state** is reached (terminal state with no outgoing transitions)
2. A **convergence function** returns `True` (custom quality criterion)
3. The **maximum steps** limit is hit (safety ceiling)

In [ ]:
# SDK Imports — everything we need for MCMC state machines
import asyncio
import random
from multigen import StateMachine, Sampler, EnsembleResult, FunctionAgent

print("Multigen SDK imports loaded successfully.")
print("Ready to build MCMC probabilistic state machines!")

## Section 4: Building a 3-State Writing Machine

Let's build the classic **draft → review → final** writing machine.

We'll define:
- Three `FunctionAgent` instances — one per state
- A `StateMachine` that orchestrates transitions
- Probabilities that model realistic quality-gating behavior

### The Three Agents

Each agent represents a stage in the writing workflow:
- **draft_agent**: Generates initial content, may produce rough output
- **review_agent**: Evaluates and improves content, checks quality criteria
- **final_agent**: Polishes and prepares for publication

In [ ]:
# --- Define the three writing stage agents ---

# Draft agent: generates initial content
# In production this would call an LLM
async def draft_fn(input_text, context=None):
    """
    Simulates a drafting agent.
    Returns a rough draft with some quality score.
    Quality is random to simulate the stochastic nature of LLM output.
    """
    quality = random.uniform(0.4, 0.95)  # Simulate variable LLM quality
    return {
        "content": f"[DRAFT] Initial draft of: {input_text}",
        "quality_score": quality,
        "word_count": random.randint(200, 800),
        "stage": "draft"
    }

# Review agent: evaluates and improves the draft
async def review_fn(draft_result, context=None):
    """
    Simulates a review/editing agent.
    Takes draft content, improves it, returns revised version.
    The review stage typically improves quality by 10-20%.
    """
    if isinstance(draft_result, dict):
        prev_quality = draft_result.get("quality_score", 0.6)
        content = draft_result.get("content", str(draft_result))
    else:
        prev_quality = 0.6
        content = str(draft_result)
    
    # Review improves quality but never perfectly
    new_quality = min(0.99, prev_quality + random.uniform(0.05, 0.20))
    return {
        "content": f"[REVIEWED] {content} → Improved and fact-checked.",
        "quality_score": new_quality,
        "review_notes": "Grammar improved, facts verified, structure tightened.",
        "stage": "review"
    }

# Final agent: polishes for publication
async def final_fn(reviewed_result, context=None):
    """
    Simulates a final polish/publication agent.
    Applies formatting, adds metadata, prepares for output.
    """
    if isinstance(reviewed_result, dict):
        content = reviewed_result.get("content", str(reviewed_result))
        quality = reviewed_result.get("quality_score", 0.8)
    else:
        content = str(reviewed_result)
        quality = 0.8
    
    return {
        "content": f"[FINAL] {content} → Ready for publication.",
        "quality_score": quality,
        "published": True,
        "stage": "final",
        "format": "markdown"
    }

# Create FunctionAgent instances from the async functions
draft_agent  = FunctionAgent(draft_fn,  name="DraftAgent")
review_agent = FunctionAgent(review_fn, name="ReviewAgent")
final_agent  = FunctionAgent(final_fn,  name="FinalAgent")

print("Three writing stage agents created:")
print(f"  draft_agent  → {draft_agent.name}")
print(f"  review_agent → {review_agent.name}")
print(f"  final_agent  → {final_agent.name}")

## Section 5: Transition Probabilities

Now we wire the agents into a `StateMachine` and define the transition probabilities.

The `sm.transition(from_state, to_state, prob=...)` method adds a directed edge in the state graph.

**Design philosophy:**
- A state can have multiple outgoing transitions that sum to 1.0
- Probabilities can be static (set once) or dynamic (functions of runtime context)
- Missing transitions from a state make it an absorbing (terminal) state

In [ ]:
# --- Build the StateMachine ---

# Create the state machine with a name and initial state
sm = StateMachine(
    name="WritingMachine",
    initial_state="draft",
    max_steps=20,          # Safety ceiling: at most 20 transitions
    sampler=Sampler.WEIGHTED,  # Use probability-weighted sampling
    temperature=1.0        # Standard temperature — use raw probabilities
)

# --- Register states with their agents ---
# Each state name maps to an agent that runs when that state is entered
sm.state("draft",  draft_agent)
sm.state("review", review_agent)
sm.state("final",  final_agent)  # Terminal state — no outgoing transitions from here

# --- Define transitions with probabilities ---

# From DRAFT:
#   80% chance → go to REVIEW (draft is good enough to review)
#   20% chance → back to DRAFT (self-loop: draft needs another pass)
sm.transition("draft", "review", prob=0.8)
sm.transition("draft", "draft",  prob=0.2)   # Self-loop: revision

# From REVIEW:
#   90% chance → go to FINAL (review passed, ready to publish)
#   10% chance → back to DRAFT (major issues found, needs complete rewrite)
sm.transition("review", "final", prob=0.9)
sm.transition("review", "draft", prob=0.1)   # Send back for major revision

# Note: "final" has no outgoing transitions → it is an absorbing/terminal state
# The machine will STOP when it reaches "final"

print("StateMachine configured:")
print(f"  Name:          {sm.name}")
print(f"  Initial state: {sm.initial_state}")
print(f"  States:        {list(sm.states.keys())}")
print(f"  Sampler:       {sm.sampler}")
print(f"  Temperature:   {sm.temperature}")
print(f"  Max steps:     {sm.max_steps}")
print()
print("Transition summary:")
print("  draft  → review (0.8)")
print("  draft  → draft  (0.2)  [self-loop]")
print("  review → final  (0.9)")
print("  review → draft  (0.1)  [back to draft]")
print("  final  → [TERMINAL]")

## Section 6: Running the Machine

Use `await sm.run(ctx)` to execute the state machine from its initial state.

The result object has three key fields:
- **`result.path`**: The sequence of states visited (e.g., `['draft', 'draft', 'review', 'final']`)
- **`result.status`**: Whether execution completed normally or hit `max_steps`
- **`result.final_output`**: The output from the terminal state's agent

Each run may follow a different path because transitions are sampled stochastically!

In [ ]:
# --- Run the state machine ---

async def run_writing_machine():
    """
    Executes the writing state machine from its initial state.
    The machine runs until it reaches the terminal 'final' state.
    """
    # The initial input to the machine (passed to the first state's agent)
    initial_input = "Write a technical article about MCMC methods in AI."
    
    print(f"Input: '{initial_input}'")
    print(f"Starting state: '{sm.initial_state}'")
    print("-" * 60)
    
    # Run the machine — this is the main execution call
    # 'ctx' can be an AgentContext or a plain dict; here we pass the input string
    result = await sm.run(initial_input)
    
    return result

# Execute and inspect the result
result = asyncio.run(run_writing_machine())

print()
print("=" * 60)
print("EXECUTION RESULT")
print("=" * 60)

# result.path — the sequence of states visited
print(f"result.path:         {result.path}")
# Example output: ['draft', 'review', 'final']
# Or with a revision: ['draft', 'draft', 'review', 'final']

# result.status — 'completed' if terminal state reached, 'max_steps' if ceiling hit
print(f"result.status:       {result.status}")

# result.steps — total number of state transitions
print(f"result.steps:        {result.steps}")

# result.final_output — output from the last (terminal) state's agent
print(f"result.final_output: {result.final_output}")

print()
print("Path length:", len(result.path), "states visited")
print("Unique states:", sorted(set(result.path)))

# Count how many times draft was revisited
draft_count = result.path.count('draft')
print(f"Draft revisions: {draft_count} (self-loops or revisits)")

## Section 7: Self-Loops — Draft Back to Draft

The self-loop `draft → draft (prob=0.2)` models the scenario where a draft needs more work before it's ready for review.

This is a powerful concept: **the machine can revisit a state multiple times** until it produces output good enough to move forward.

Let's run the machine 10 times and observe how often revisions occur:

In [ ]:
# --- Demonstrate self-loop revision behavior across multiple runs ---

async def run_multiple_times(n=10):
    """
    Run the state machine n times and collect path statistics.
    This shows the stochastic variability in trajectories.
    """
    paths = []
    for i in range(n):
        result = await sm.run("Article input")
        paths.append(result.path)
    return paths

paths = asyncio.run(run_multiple_times(10))

print("=" * 60)
print("10 INDEPENDENT RUNS — OBSERVING TRAJECTORY VARIABILITY")
print("=" * 60)

for i, path in enumerate(paths):
    draft_count = path.count('draft')
    total_steps = len(path)
    had_self_loop = draft_count > 1 or (draft_count == 1 and path[0] == 'draft' and path[1] == 'draft' if len(path) > 1 else False)
    
    # Build a compact representation
    path_str = " → ".join(path)
    
    print(f"Run {i+1:2d}: {path_str}")
    print(f"        Steps: {total_steps}, Draft visits: {draft_count}")

print()
# Compute statistics
avg_steps = sum(len(p) for p in paths) / len(paths)
avg_draft_visits = sum(p.count('draft') for p in paths) / len(paths)
direct_paths = sum(1 for p in paths if p == ['draft', 'review', 'final'])

print(f"Average steps per run:        {avg_steps:.1f}")
print(f"Average draft visits per run: {avg_draft_visits:.1f}")
print(f"Direct paths (no revision):   {direct_paths}/{len(paths)}")
print()
print("Observation: Because draft→draft has prob=0.2 and draft→review has prob=0.8,")
print("roughly 20% of steps from 'draft' will loop back — creating revision cycles.")

## Section 8: MCMC Samplers — weighted, greedy, metropolis

The SDK provides three sampler strategies via the `Sampler` enum. Each implements a different method for selecting the next state from the transition distribution.

### Sampler.WEIGHTED
Sample proportional to transition probabilities using `numpy.random.choice` or equivalent weighted sampling.
```
P(choose j) = prob(i→j) / sum_k prob(i→k)
```
Best for general use — mirrors how you'd intuitively think about probabilistic transitions.

### Sampler.GREEDY
Always pick the transition with the highest probability. Equivalent to temperature=0.0.
```
next_state = argmax_j prob(i→j)
```
Produces **deterministic, reproducible** paths. Useful in production when you want predictability.

### Sampler.METROPOLIS
Implements the Metropolis-Hastings accept/reject algorithm:
1. Propose a candidate next state `j` uniformly at random from possible transitions
2. Compute acceptance ratio `α = P(i→j) / P(i→current)` (or use log probs)
3. Accept with probability `min(1, α)`, otherwise stay

This gives theoretically correct sampling from the stationary distribution, useful when transition probabilities span many orders of magnitude.

In [ ]:
# --- Compare the three samplers on the same state machine ---

async def compare_samplers():
    """
    Create three identical state machines with different samplers
    and run each 5 times to compare path distributions.
    """
    samplers = [
        (Sampler.WEIGHTED,    "WEIGHTED   "),
        (Sampler.GREEDY,      "GREEDY     "),
        (Sampler.METROPOLIS,  "METROPOLIS ")
    ]
    
    results = {}
    
    for sampler_type, sampler_name in samplers:
        # Create a fresh StateMachine with this sampler
        machine = StateMachine(
            name=f"WritingMachine_{sampler_name.strip()}",
            initial_state="draft",
            max_steps=20,
            sampler=sampler_type,
            temperature=1.0
        )
        machine.state("draft",  draft_agent)
        machine.state("review", review_agent)
        machine.state("final",  final_agent)
        machine.transition("draft",  "review", prob=0.8)
        machine.transition("draft",  "draft",  prob=0.2)
        machine.transition("review", "final",  prob=0.9)
        machine.transition("review", "draft",  prob=0.1)
        
        paths = []
        for _ in range(5):
            r = await machine.run("Article input")
            paths.append(r.path)
        
        results[sampler_name] = paths
    
    return results

sampler_results = asyncio.run(compare_samplers())

print("=" * 70)
print("SAMPLER COMPARISON — 5 runs each")
print("=" * 70)

for sampler_name, paths in sampler_results.items():
    print(f"\n[{sampler_name}]")
    for i, path in enumerate(paths):
        print(f"  Run {i+1}: {' → '.join(path)}  ({len(path)} steps)")
    
    avg = sum(len(p) for p in paths) / len(paths)
    print(f"  Average steps: {avg:.1f}")

print()
print("Key observations:")
print("  GREEDY     → Always takes the highest-prob transition: draft→review→final")
print("               All 5 runs should produce identical paths.")
print("  WEIGHTED   → Samples proportionally, variability expected.")
print("  METROPOLIS → Accept/reject sampling, may show different variability.")

## Section 9: Temperature Effect

Temperature controls the **entropy** of the transition distribution.

At **temperature 0.0**, the distribution collapses to a one-hot (greedy). At **temperature 2.0**, probabilities become more uniform — the machine explores more paths.

This is identical to LLM sampling temperature: low temperature = conservative, high = creative.

Let's compare the same machine at T=0.0, T=1.0, and T=2.0:

In [ ]:
# --- Temperature comparison ---

async def compare_temperatures():
    """
    Run the same state machine at three different temperatures.
    T=0.0 → greedy (deterministic)
    T=1.0 → standard sampling
    T=2.0 → high entropy (more exploration)
    """
    temperatures = [0.0, 1.0, 2.0]
    results = {}
    
    for temp in temperatures:
        machine = StateMachine(
            name=f"WritingMachine_T{temp}",
            initial_state="draft",
            max_steps=20,
            sampler=Sampler.WEIGHTED,
            temperature=temp
        )
        machine.state("draft",  draft_agent)
        machine.state("review", review_agent)
        machine.state("final",  final_agent)
        machine.transition("draft",  "review", prob=0.8)
        machine.transition("draft",  "draft",  prob=0.2)
        machine.transition("review", "final",  prob=0.9)
        machine.transition("review", "draft",  prob=0.1)
        
        paths = []
        for _ in range(8):  # Run 8 times to see distribution
            r = await machine.run("Article input")
            paths.append(r.path)
        
        results[temp] = paths
    
    return results

temp_results = asyncio.run(compare_temperatures())

print("=" * 70)
print("TEMPERATURE COMPARISON — 8 runs each")
print("=" * 70)

for temp, paths in temp_results.items():
    unique_paths = set(tuple(p) for p in paths)
    avg_steps = sum(len(p) for p in paths) / len(paths)
    max_steps = max(len(p) for p in paths)
    
    print(f"\nTemperature = {temp}")
    for path in paths:
        print(f"  {' → '.join(path)}")
    print(f"  Unique trajectories: {len(unique_paths)} / {len(paths)} runs")
    print(f"  Average steps:       {avg_steps:.1f}")
    print(f"  Max steps in run:    {max_steps}")

print()
print("Expected behavior:")
print("  T=0.0 → All 8 runs identical (greedy: always draft→review→final)")
print("  T=1.0 → Some variability, occasional self-loops")
print("  T=2.0 → More variability, more self-loops, longer paths on average")

## Section 10: Adaptive Transitions

Static transition probabilities are great for fixed workflows. But for quality-driven workflows, we want transitions that **adapt based on the agent's actual output quality**.

The `sm.adaptive_transition(from_state, to_state, score_fn, threshold)` method sets up a transition where:
- `score_fn(output)` evaluates the quality of the current state's output
- If `score_fn(output) >= threshold`: the transition is taken (probability effectively 1.0)
- If `score_fn(output) < threshold`: the transition is skipped, and the machine tries other options

This enables **quality-gated state machines** that won't advance until quality criteria are met.

In [ ]:
# --- Adaptive transitions based on output quality ---

# Define quality scoring functions
def review_quality_score(output):
    """
    Evaluates whether a draft is ready for review.
    In production, this might call a judge LLM or run heuristics.
    Returns a float in [0.0, 1.0].
    """
    if isinstance(output, dict):
        return output.get("quality_score", 0.5)
    return 0.5  # Fallback if output format is unexpected

def final_quality_score(output):
    """
    Evaluates whether a reviewed draft is ready to publish.
    Stricter threshold than draft→review.
    """
    if isinstance(output, dict):
        quality = output.get("quality_score", 0.5)
        # Also check it's been reviewed (not just the raw draft)
        is_reviewed = output.get("stage") == "review"
        return quality if is_reviewed else quality * 0.7
    return 0.5

# Build an adaptive state machine
adaptive_sm = StateMachine(
    name="AdaptiveWritingMachine",
    initial_state="draft",
    max_steps=15,
    sampler=Sampler.WEIGHTED,
    temperature=1.0
)

adaptive_sm.state("draft",  draft_agent)
adaptive_sm.state("review", review_agent)
adaptive_sm.state("final",  final_agent)

# Adaptive transition: draft → review only if quality_score >= 0.65
# If score < 0.65, this transition won't fire and the machine stays in draft (implicit self-loop)
adaptive_sm.adaptive_transition(
    from_state="draft",
    to_state="review",
    score_fn=review_quality_score,
    threshold=0.65
)

# Adaptive transition: review → final only if quality_score >= 0.80
adaptive_sm.adaptive_transition(
    from_state="review",
    to_state="final",
    score_fn=final_quality_score,
    threshold=0.80
)

# Fallback: review → draft if quality score is too low
adaptive_sm.transition("review", "draft", prob=0.15)

print("Adaptive state machine configured.")
print("Transitions:")
print("  draft  → review  [ADAPTIVE: score_fn >= 0.65]")
print("  review → final   [ADAPTIVE: score_fn >= 0.80]")
print("  review → draft   [FALLBACK: prob=0.15]")
print()

async def run_adaptive():
    result = await adaptive_sm.run("Write an article about adaptive workflows.")
    return result

adaptive_result = asyncio.run(run_adaptive())
print(f"Path:   {adaptive_result.path}")
print(f"Status: {adaptive_result.status}")
print(f"Steps:  {adaptive_result.steps}")

## Section 11: transition_matrix()

The `sm.transition_matrix()` method returns the **normalized probability matrix** for all states in the machine.

This is useful for:
- **Debugging**: Verify that probabilities are set correctly
- **Visualization**: Plot the graph with edge weights
- **Analysis**: Compute stationary distribution, expected path length, etc.

The returned matrix is a dictionary-of-dictionaries where `matrix[from_state][to_state]` gives the transition probability.

In [ ]:
# --- Inspect the transition matrix ---

# Get the normalized transition matrix
matrix = sm.transition_matrix()

print("=" * 60)
print("TRANSITION MATRIX — WritingMachine")
print("=" * 60)
print()

# Get all state names for headers
states = list(sm.states.keys())

# Print header row
header = "From\\To  " + "  ".join(f"{s:10s}" for s in states)
print(header)
print("-" * len(header))

# Print each row
for from_state in states:
    row = f"{from_state:8s}  "
    for to_state in states:
        prob = matrix.get(from_state, {}).get(to_state, 0.0)
        row += f"{prob:10.3f}  "
    print(row)

print()
print("Row sums (should all be 1.0 except terminal states):")
for from_state in states:
    row_sum = sum(matrix.get(from_state, {}).values())
    terminal = "[TERMINAL]" if row_sum == 0 else ""
    print(f"  {from_state}: {row_sum:.3f} {terminal}")

print()
print("Matrix interpretation:")
print("  draft  → draft:  0.200 (20% self-loop: needs revision)")
print("  draft  → review: 0.800 (80% advance to review)")
print("  review → draft:  0.100 (10% sent back for rewrite)")
print("  review → final:  0.900 (90% advance to publication)")
print("  final  → [none]: 0.000 (absorbing state, machine stops)")

## Section 12: run_ensemble() — 5 Parallel Chains

One of the most powerful features of the MCMC state machine is **ensemble execution**: running `N` independent chains in parallel and picking the best result.

`sm.run_ensemble(chains=N)` runs the state machine N times concurrently (using asyncio tasks) and returns an `EnsembleResult` with:

- **`ensemble.best_path`**: The path from the chain with the highest-scoring final output
- **`ensemble.agreement_score`**: Float [0,1] — how much the chains agree on the final path
- **`ensemble.consensus_output`**: The most common final output across chains
- **`ensemble.all_results`**: List of individual run results
- **`ensemble.path_histogram`**: Frequency of each unique path

Ensemble execution is analogous to **majority voting** or **best-of-N** sampling in LLMs.

In [ ]:
# --- Run ensemble of 5 parallel chains ---

async def run_ensemble_demo():
    """
    Run 5 independent chains of the state machine in parallel.
    Returns an EnsembleResult with aggregated statistics.
    """
    print("Running 5 parallel chains...")
    print("Each chain independently explores the state space.")
    print()
    
    ensemble = await sm.run_ensemble(
        input_data="Write a comprehensive guide to ensemble methods.",
        chains=5
    )
    
    return ensemble

ensemble = asyncio.run(run_ensemble_demo())

print("=" * 60)
print("ENSEMBLE RESULT (5 chains)")
print("=" * 60)
print()

# Individual chain results
print("Individual chain paths:")
for i, result in enumerate(ensemble.all_results):
    print(f"  Chain {i+1}: {' → '.join(result.path)}  ({result.steps} steps)")

print()

# Best path — from the chain with highest quality final output
print(f"ensemble.best_path:       {ensemble.best_path}")
# e.g., ['draft', 'review', 'final'] — the most efficient high-quality path

# Agreement score — how similar the paths were across chains
# 1.0 = all chains took identical paths
# 0.0 = all chains took completely different paths
print(f"ensemble.agreement_score: {ensemble.agreement_score:.3f}")

# Consensus output — the final output that appeared most frequently
print(f"ensemble.consensus_output: (truncated)")
if isinstance(ensemble.consensus_output, dict):
    print(f"  stage:         {ensemble.consensus_output.get('stage')}")
    print(f"  quality_score: {ensemble.consensus_output.get('quality_score', 'N/A')}")
    print(f"  published:     {ensemble.consensus_output.get('published')}")

print()
print("Path histogram (unique paths across all chains):")
for path_tuple, count in sorted(ensemble.path_histogram.items(), key=lambda x: -x[1]):
    path_str = " → ".join(path_tuple)
    bar = "█" * count
    print(f"  {path_str}")
    print(f"  {bar} ({count}/{len(ensemble.all_results)} chains)")

In [ ]:
# --- Larger ensemble: 10 chains ---

async def run_large_ensemble():
    """
    Run 10 chains to get more reliable statistics.
    With more chains, the agreement_score and path histogram are more meaningful.
    """
    ensemble = await sm.run_ensemble(
        input_data="Technical article on MCMC ensemble methods.",
        chains=10
    )
    return ensemble

large_ensemble = asyncio.run(run_large_ensemble())

print("=" * 60)
print("LARGE ENSEMBLE RESULT (10 chains)")
print("=" * 60)

print(f"\nBest path:       {large_ensemble.best_path}")
print(f"Agreement score: {large_ensemble.agreement_score:.3f}")
print(f"Total chains:    {len(large_ensemble.all_results)}")

steps_list = [r.steps for r in large_ensemble.all_results]
print(f"Min steps: {min(steps_list)}, Max steps: {max(steps_list)}, Avg: {sum(steps_list)/len(steps_list):.1f}")

print("\nPath distribution:")
for path_tuple, count in sorted(large_ensemble.path_histogram.items(), key=lambda x: -x[1]):
    pct = count / len(large_ensemble.all_results) * 100
    print(f"  {' → '.join(path_tuple):50s}  {count:2d}/10  ({pct:.0f}%)")

## Section 13: Convergence Detection

By default, the state machine stops when it reaches an absorbing state (no outgoing transitions) or hits `max_steps`.

For more sophisticated control, you can define a **custom convergence function** via `sm.set_convergence(fn)`. This function receives the current execution context and returns `True` when the machine should stop.

This is useful when:
- You want to stop after quality exceeds a threshold (regardless of which state you're in)
- You want to stop after a certain number of revisions
- You have domain-specific stopping criteria

In [ ]:
# --- Custom convergence detection ---

# Convergence function signature: fn(state, output, step_count) → bool
# Return True to stop the machine immediately

def quality_convergence(state, output, step_count):
    """
    Stop the machine when:
    1. We're in the 'review' state AND quality is above 0.85, OR
    2. We've taken more than 10 steps (avoid infinite loops)
    
    This allows early termination if the review is excellent —
    we don't need to go through 'final' formatting if quality is already perfect.
    """
    # Safety check: too many steps
    if step_count > 10:
        print(f"  [convergence] Stopping: exceeded 10 steps (at step {step_count})")
        return True
    
    # Quality-based early termination
    if state == "review" and isinstance(output, dict):
        quality = output.get("quality_score", 0.0)
        if quality >= 0.92:
            print(f"  [convergence] Stopping: review quality {quality:.3f} >= 0.92 threshold")
            return True
    
    return False  # Continue execution

# Create a machine with custom convergence
converging_sm = StateMachine(
    name="ConvergingWritingMachine",
    initial_state="draft",
    max_steps=20,
    sampler=Sampler.WEIGHTED,
    temperature=1.0
)

converging_sm.state("draft",  draft_agent)
converging_sm.state("review", review_agent)
converging_sm.state("final",  final_agent)
converging_sm.transition("draft",  "review", prob=0.8)
converging_sm.transition("draft",  "draft",  prob=0.2)
converging_sm.transition("review", "final",  prob=0.9)
converging_sm.transition("review", "draft",  prob=0.1)

# Register the custom convergence function
converging_sm.set_convergence(quality_convergence)

print("Convergence function registered.")
print("Machine will stop early if review quality >= 0.92 or > 10 steps.")
print()

async def run_with_convergence():
    results = []
    for i in range(5):
        print(f"Run {i+1}:")
        r = await converging_sm.run("Article with convergence detection.")
        results.append(r)
        print(f"  Path: {' → '.join(r.path)}, Status: {r.status}")
    return results

convergence_results = asyncio.run(run_with_convergence())

print()
print("Summary:")
converged_early = sum(1 for r in convergence_results if r.status == 'converged')
completed = sum(1 for r in convergence_results if r.status == 'completed')
print(f"  Converged early (quality threshold): {converged_early}")
print(f"  Completed normally (terminal state): {completed}")

## Section 14: On-Enter / On-Exit Hooks

Every state in the machine can have **lifecycle hooks** — functions that run when the machine enters or exits that state:

- `on_enter(state, input_data)` — called just before the state's agent runs
- `on_exit(state, output_data)` — called just after the state's agent completes

Hooks are useful for:
- **Logging and tracing** — record when each state is entered/exited
- **Side effects** — send notifications, update databases
- **Input transformation** — preprocess input before the agent sees it
- **Quality checks** — run lightweight validation before committing to the transition

In [ ]:
# --- On-enter and on-exit hooks ---

# Track all hook events for inspection
hook_events = []

def on_enter_review(state_name, input_data):
    """
    Called when the machine enters the 'review' state.
    Good place to log, notify, or preprocess.
    """
    quality = input_data.get('quality_score', '?') if isinstance(input_data, dict) else '?'
    msg = f"[HOOK] Entering REVIEW — incoming quality: {quality}"
    hook_events.append(msg)
    print(msg)
    # Could also: send Slack notification, increment Prometheus counter, etc.

def on_exit_review(state_name, output_data):
    """
    Called when the machine exits the 'review' state.
    Good place to validate the review agent's output.
    """
    quality = output_data.get('quality_score', '?') if isinstance(output_data, dict) else '?'
    notes = output_data.get('review_notes', '') if isinstance(output_data, dict) else ''
    msg = f"[HOOK] Exiting REVIEW — final quality: {quality}, notes: {notes[:50]}"
    hook_events.append(msg)
    print(msg)

def on_enter_final(state_name, input_data):
    """Called when entering the terminal 'final' state."""
    msg = "[HOOK] Entering FINAL — publication imminent!"
    hook_events.append(msg)
    print(msg)

# Build machine with hooks
hooked_sm = StateMachine(
    name="HookedWritingMachine",
    initial_state="draft",
    max_steps=15,
    sampler=Sampler.WEIGHTED,
    temperature=1.0
)

hooked_sm.state("draft", draft_agent)   # No hooks on draft

# Register hooks on the 'review' state
hooked_sm.state(
    "review",
    review_agent,
    on_enter=on_enter_review,  # Fires before review_agent.run()
    on_exit=on_exit_review     # Fires after review_agent.run()
)

# Register only on_enter hook for 'final'
hooked_sm.state(
    "final",
    final_agent,
    on_enter=on_enter_final
)

hooked_sm.transition("draft",  "review", prob=0.8)
hooked_sm.transition("draft",  "draft",  prob=0.2)
hooked_sm.transition("review", "final",  prob=0.9)
hooked_sm.transition("review", "draft",  prob=0.1)

print("Hooked state machine configured.")
print()

async def run_hooked():
    result = await hooked_sm.run("Article with lifecycle hooks.")
    return result

hooked_result = asyncio.run(run_hooked())

print()
print(f"Path: {' → '.join(hooked_result.path)}")
print()
print(f"Total hook events fired: {len(hook_events)}")

## Section 15: Real-World Example — Multi-Stage Research Refinement Machine

Let's build a realistic 4-state research workflow:

```
                    0.15 (insufficient search results)
                   ┌─────────┐
                   │         ▼
  START ──► [SEARCH] ── 0.85 ──► [ANALYZE] ── 0.75 ──► [SYNTHESIZE] ── 0.90 ──► [PUBLISH]
                                     │                      │
                                     │ 0.25 (back to search) │ 0.10 (back to analyze)
                                     └──────────────────────┘
```

We'll run an **ensemble of 10 chains** to find the best research synthesis.

In [ ]:
# --- Research refinement pipeline agents ---

async def search_fn(query, context=None):
    """Simulates a web search or knowledge base query agent."""
    coverage = random.uniform(0.3, 0.95)
    num_results = random.randint(3, 20)
    return {
        "query": str(query)[:80],
        "num_results": num_results,
        "coverage_score": coverage,
        "snippets": [f"Result {i+1}: relevant content about {query}" for i in range(min(3, num_results))],
        "stage": "search"
    }

async def analyze_fn(search_results, context=None):
    """Analyzes search results and extracts key insights."""
    if isinstance(search_results, dict):
        prev_score = search_results.get("coverage_score", 0.5)
    else:
        prev_score = 0.5
    
    analysis_quality = min(0.99, prev_score * random.uniform(0.85, 1.15))
    return {
        "insights": ["Key insight 1", "Key insight 2", "Key insight 3"],
        "analysis_depth": random.choice(["shallow", "moderate", "deep"]),
        "quality_score": analysis_quality,
        "stage": "analyze"
    }

async def synthesize_fn(analysis, context=None):
    """Synthesizes analysis into a coherent narrative."""
    if isinstance(analysis, dict):
        depth = analysis.get("analysis_depth", "moderate")
        base_quality = analysis.get("quality_score", 0.6)
        depth_bonus = {"shallow": 0.0, "moderate": 0.1, "deep": 0.2}.get(depth, 0.1)
    else:
        base_quality, depth_bonus = 0.6, 0.1
    
    synthesis_quality = min(0.99, base_quality + depth_bonus + random.uniform(-0.05, 0.15))
    return {
        "synthesis": "A comprehensive synthesis of the research findings...",
        "quality_score": synthesis_quality,
        "word_count": random.randint(500, 2000),
        "citations": random.randint(5, 25),
        "stage": "synthesize"
    }

async def publish_fn(synthesis, context=None):
    """Prepares and publishes the final research output."""
    if isinstance(synthesis, dict):
        quality = synthesis.get("quality_score", 0.7)
        citations = synthesis.get("citations", 0)
    else:
        quality, citations = 0.7, 0
    
    return {
        "title": "Research Report: MCMC Methods in Agentic AI",
        "quality_score": quality,
        "citations": citations,
        "published": True,
        "format": "PDF + Web",
        "stage": "publish"
    }

# Create agents
search_agent    = FunctionAgent(search_fn,    name="SearchAgent")
analyze_agent   = FunctionAgent(analyze_fn,   name="AnalyzeAgent")
synthesize_agent = FunctionAgent(synthesize_fn, name="SynthesizeAgent")
publish_agent   = FunctionAgent(publish_fn,   name="PublishAgent")

print("Research pipeline agents created:")
print(f"  {search_agent.name}, {analyze_agent.name}, {synthesize_agent.name}, {publish_agent.name}")

In [ ]:
# --- Build and run the research state machine with 10-chain ensemble ---

research_sm = StateMachine(
    name="ResearchRefinementMachine",
    initial_state="search",
    max_steps=25,
    sampler=Sampler.WEIGHTED,
    temperature=1.0
)

# Register states
research_sm.state("search",     search_agent)
research_sm.state("analyze",    analyze_agent)
research_sm.state("synthesize", synthesize_agent)
research_sm.state("publish",    publish_agent)

# Transitions:
# search → search (self-loop: insufficient results, search again)
research_sm.transition("search",     "search",     prob=0.15)
# search → analyze (results good enough to analyze)
research_sm.transition("search",     "analyze",    prob=0.85)
# analyze → search (results were insufficient for deep analysis)
research_sm.transition("analyze",    "search",     prob=0.25)
# analyze → synthesize (analysis complete)
research_sm.transition("analyze",    "synthesize", prob=0.75)
# synthesize → analyze (synthesis revealed gaps in analysis)
research_sm.transition("synthesize", "analyze",    prob=0.10)
# synthesize → publish (synthesis complete, ready to publish)
research_sm.transition("synthesize", "publish",    prob=0.90)
# publish is terminal (no outgoing transitions)

print("Research machine configured with 4 states:")
print("  search (self=0.15, analyze=0.85)")
print("  analyze (search=0.25, synthesize=0.75)")
print("  synthesize (analyze=0.10, publish=0.90)")
print("  publish [TERMINAL]")
print()

# Run 10-chain ensemble
async def run_research_ensemble():
    print("Running 10-chain ensemble for research pipeline...")
    print()
    
    ensemble = await research_sm.run_ensemble(
        input_data="MCMC methods in probabilistic agentic AI systems — comprehensive survey",
        chains=10
    )
    return ensemble

research_ensemble = asyncio.run(run_research_ensemble())

print("=" * 70)
print("RESEARCH ENSEMBLE RESULT (10 chains)")
print("=" * 70)
print()

print("Individual chain paths:")
for i, r in enumerate(research_ensemble.all_results):
    print(f"  Chain {i+1:2d}: {' → '.join(r.path)}")

print()
print(f"Best path:        {research_ensemble.best_path}")
print(f"Agreement score:  {research_ensemble.agreement_score:.3f}")
print(f"(1.0 = all chains identical, 0.0 = all chains different)")

print()
print("Best chain final output:")
output = research_ensemble.consensus_output
if isinstance(output, dict):
    for k, v in output.items():
        print(f"  {k}: {v}")

print()
print("Path frequency distribution:")
for path_tuple, count in sorted(research_ensemble.path_histogram.items(), key=lambda x: -x[1]):
    pct = count / 10 * 100
    print(f"  {' → '.join(path_tuple)}: {count}/10 ({pct:.0f}%)")

## Summary and Key Takeaways

In this notebook we built and ran **MCMC probabilistic state machines** for agentic workflows:

### Core Concepts Mastered

| Concept | What You Learned |
|---|---|
| **Markov chains** | States, transition probabilities, absorbing states |
| **StateMachine** | `sm.state()`, `sm.transition()`, `await sm.run()` |
| **Self-loops** | How to model revision behavior with `transition(s, s, prob=p)` |
| **Samplers** | WEIGHTED, GREEDY, METROPOLIS — trade-offs between exploration and exploitation |
| **Temperature** | How T=0 (greedy) vs T=2 (exploratory) affects path diversity |
| **Adaptive transitions** | Quality-gated advancement with `adaptive_transition()` |
| **Transition matrix** | Inspecting the full probability matrix with `transition_matrix()` |
| **Ensemble runs** | `run_ensemble(chains=N)` → `best_path`, `agreement_score`, `consensus_output` |
| **Convergence** | Custom stopping criteria with `set_convergence(fn)` |
| **Hooks** | State lifecycle with `on_enter`, `on_exit` |

### When to Use MCMC State Machines

Use `StateMachine` when your workflow:
- Needs **quality-driven iteration** (retry until good enough)
- Has **non-linear paths** (sometimes skip stages, sometimes revisit them)
- Benefits from **ensemble exploration** (run 10 variants, pick best)
- Requires **principled termination** (convergence functions, not ad-hoc counters)

### Next Steps

- **Notebook 17**: Event Bus and Messaging — Pub/Sub, Request/Reply, Actor Patterns
- **Notebook 18**: Runtime — Unified Execution with Observability
- **Notebook 19**: Resilience — Circuit Breakers, Retry, Memory, Human-in-the-Loop